<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/stress_mapping_gene_fxns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
!pip install -q requests pandas

import io, re, time
import requests
import pandas as pd

pd.set_option('display.max_colwidth', 100)

BIOMART = 'https://plants.ensembl.org/biomart/martservice'
CROPS   = ['soy', 'chilli', 'cotton', 'grape', 'tomato']

# Verified working dataset names. Grape and cotton are NOT here:
#   grape  -> VITVI* is the PN40024.v4 annotation; Ensembl has moved to a
#             different assembly and no longer serves these IDs. File route (Cell 7).
#   cotton -> LOC* are NCBI GeneIDs, not Ensembl. UniProt route (Cell 5).
DATASETS = {
    'soy':    'gmax_eg_gene',
    'chilli': 'cannuum_eg_gene',
}
print('ready')

ready


In [36]:
from google.colab import files

uploaded = files.upload()
CSV = list(uploaded.keys())[0]

df = pd.read_csv(CSV)
print(f'{CSV}: {df.shape[0]} peptides')
df.head(3)

Saving common_peptides_stress.csv to common_peptides_stress (2).csv
common_peptides_stress (2).csv: 286 peptides


,peptide_id,peptide_seq,n_crops,soy,chilli,cotton,grape,tomato
0,ATQ66586.1_pep72_mc0,RRRRRE,5,GLYMA_02G118500;GLYMA_02G294100;GLYMA_04G039300;GLYMA_04G190100;GLYMA_05G063500;GLYMA_06G040400;...,T459_06263;T459_07223,LOC107889151;LOC107891587;LOC107912366;LOC107922766;LOC107937850,VITVI02G01498;VITVI04G01535;VITVI06G00459;VITVI11G00037;VITVI11G00227;VITVI14G00195;VITVI17G0089...,SOLYC10G079420.1;SOLYC11G073110.2
1,ATQ68623.1_pep110_mc0,RRRGRL,5,GLYMA_01G008600;GLYMA_13G104500;GLYMA_14G041500,T459_00721;T459_07270;T459_18026;T459_18027,LOC107890504;LOC107920344;LOC107944210,VITVI02G00919;VITVI06G00126;VITVI10G00198,SOLYC06G073730.2;SOLYC08G007220.3
2,ATQ66727.1_pep51_mc0,RRRRRE,5,GLYMA_02G118500;GLYMA_02G294100;GLYMA_04G039300;GLYMA_04G190100;GLYMA_05G063500;GLYMA_06G040400;...,T459_06263;T459_07223,LOC107889151;LOC107891587;LOC107912366;LOC107922766;LOC107937850,VITVI02G01498;VITVI04G01535;VITVI06G00459;VITVI11G00037;VITVI11G00227;VITVI14G00195;VITVI17G0089...,SOLYC10G079420.1;SOLYC11G073110.2


In [37]:
rows = []
for _, r in df.iterrows():
    for crop in CROPS:
        cell = r.get(crop)
        if pd.isna(cell):
            continue
        for g in str(cell).split(';'):
            g = g.strip()
            if g:
                rows.append({'peptide_id': r.peptide_id,
                             'peptide_seq': r.peptide_seq,
                             'n_crops': r.n_crops,
                             'crop': crop,
                             'gene_id': g})

long = pd.DataFrame(rows)

def to_query_id(crop, g):
    if crop == 'tomato':                      # SOLYC01G010650.3 -> Solyc01g010650
        g = g.split('.')[0]
        m = re.match(r'^SOLYC(\d+)G(\d+)$', g)
        return f'Solyc{m.group(1)}g{m.group(2)}' if m else g
    if crop == 'cotton':                      # LOC107886984 -> 107886984
        return g.replace('LOC', '')
    return g                                  # soy, chilli, grape unchanged

long['query_id'] = [to_query_id(c, g) for c, g in zip(long.crop, long.gene_id)]

print(f'{len(long)} peptide-gene pairs | {long.gene_id.nunique()} unique genes\n')
print(long.groupby('crop').gene_id.nunique().to_string())

1762 peptide-gene pairs | 1467 unique genes

crop
chilli    280
cotton    239
grape     203
soy       564
tomato    181


In [38]:
UP     = 'https://rest.uniprot.org'
FIELDS = 'accession,gene_names,protein_name,xref_ensemblplants'

ORGANISM = {
    'soy':    3847,   # Glycine max
    'chilli': 4072,   # Capsicum annuum
    'tomato': 4081,   # Solanum lycopersicum
}


def uniprot_search(ids, organism_id, batch=40):
    """Batched text search. Returns raw UniProt rows."""
    hits = []
    for i in range(0, len(ids), batch):
        chunk = ids[i:i + batch]
        q = '(' + ' OR '.join(chunk) + f') AND organism_id:{organism_id}'
        for attempt in range(3):
            try:
                r = requests.get(f'{UP}/uniprotkb/search',
                                 params={'query': q, 'format': 'tsv',
                                         'fields': FIELDS, 'size': 500},
                                 timeout=120)
                r.raise_for_status()
                t = pd.read_csv(io.StringIO(r.text), sep='\t')
                if not t.empty:
                    hits.append(t)
                break
            except Exception as e:
                if attempt == 2:
                    print(f'  ! batch {i} failed: {e}')
                time.sleep(5)
        print(f'    {min(i+batch, len(ids))}/{len(ids)}', end='\r')
    print(' ' * 30, end='\r')
    return pd.concat(hits, ignore_index=True) if hits else pd.DataFrame()


def backmap(raw, ids, crop):
    """A text search doesn't say which ID matched a row. Scan every column to find out."""
    cols = ['crop', 'query_id', 'gene_name', 'description']
    if raw.empty:
        return pd.DataFrame(columns=cols)

    blob = raw.fillna('').astype(str)
    rows = []
    for _, row in blob.iterrows():
        text = ' '.join(row.values).upper()          # includes the Ensembl xref column
        for gid in ids:
            if gid.upper() in text:
                rows.append({'crop': crop,
                             'query_id': gid,
                             'gene_name': row.get('Gene Names', ''),
                             'description': row.get('Protein names', '')})
    return (pd.DataFrame(rows).drop_duplicates('query_id')
            if rows else pd.DataFrame(columns=cols))


up_tables = []
for crop, org in ORGANISM.items():
    ids = sorted(long.loc[long.crop == crop, 'query_id'].unique())
    print(f'[{crop}] {len(ids)} genes -> UniProt (organism_id:{org})')

    raw = uniprot_search(ids, org)
    t = backmap(raw, ids, crop)
    up_tables.append(t)

    print(f'    {len(raw)} raw rows -> {len(t)}/{len(ids)} genes matched')
    if t.empty and not raw.empty:                    # diagnose rather than fail silently
        print('    !! rows returned but none back-mapped. columns:',
              raw.columns.tolist())
        print('    !! sample row:', raw.iloc[0].to_dict())

print('\ndone')

[soy] 564 genes -> UniProt (organism_id:3847)
    927 raw rows -> 3/564 genes matched
[chilli] 280 genes -> UniProt (organism_id:4072)
    354 raw rows -> 280/280 genes matched
[tomato] 181 genes -> UniProt (organism_id:4081)
    192 raw rows -> 181/181 genes matched

done


In [39]:
soy_ids = sorted(long.loc[long.crop == 'soy', 'query_id'].unique())
print(f'[soy] {len(soy_ids)} genes -> UniProt, one at a time (~2-3 min)')

soy_rows, misses = [], []
for n, gid in enumerate(soy_ids, 1):
    for attempt in range(3):
        try:
            r = requests.get(f'{UP}/uniprotkb/search',
                             params={'query': f'{gid} AND organism_id:3847',
                                     'format': 'tsv',
                                     'fields': 'accession,gene_names,protein_name',
                                     'size': 1},
                             timeout=60)
            r.raise_for_status()
            b = r.text.strip().split('\n')
            if len(b) > 1:
                p = b[1].split('\t')
                soy_rows.append({'crop': 'soy',
                                 'query_id': gid,
                                 'gene_name': p[1] if len(p) > 1 else '',
                                 'description': p[2] if len(p) > 2 else ''})
            else:
                misses.append(gid)
            break
        except Exception:
            if attempt == 2:
                misses.append(gid)
            time.sleep(3)
    if n % 25 == 0 or n == len(soy_ids):
        print(f'    {n}/{len(soy_ids)}  (hits {len(soy_rows)})', end='\r')
    time.sleep(0.1)

soy_t = pd.DataFrame(soy_rows)
print(f'\n    {len(soy_t)}/{len(soy_ids)} matched, {len(misses)} misses')

# splice soy back in, keep the working chilli + tomato tables
up_tables = [soy_t] + up_tables[1:]
for t in up_tables:
    print(f'  {t.crop.iloc[0] if len(t) else "?":8s} {len(t)} genes')

soy_t.head(3)

[soy] 564 genes -> UniProt, one at a time (~2-3 min)
    564/564  (hits 564)
    564/564 matched, 0 misses
  soy      564 genes
  chilli   280 genes
  tomato   181 genes


,crop,query_id,gene_name,description
0,soy,GLYMA_01G005000,LOC100809761,C3H1-type domain-containing protein
1,soy,GLYMA_01G005400,LOC100793844,Phosphofructokinase domain-containing protein
2,soy,GLYMA_01G007100,LOC100798432,Protein DETOXIFICATION


In [45]:
UP = 'https://rest.uniprot.org'

cot_ids = sorted(long.loc[long.crop == 'cotton', 'query_id'].unique())
print(f'[cotton] {len(cot_ids)} genes -> UniProt idmapping')

job = requests.post(f'{UP}/idmapping/run',
                    data={'from': 'GeneID', 'to': 'UniProtKB',
                          'ids': ','.join(cot_ids)}, timeout=60)
job.raise_for_status()
jid = job.json()['jobId']

for _ in range(60):
    s = requests.get(f'{UP}/idmapping/status/{jid}', timeout=60).json()
    if s.get('jobStatus') not in ('RUNNING', 'NEW'):
        break
    time.sleep(3)

r = requests.get(f'{UP}/idmapping/uniprotkb/results/stream/{jid}',
                 params={'format': 'tsv', 'fields': 'protein_name,gene_names'},
                 timeout=300)
r.raise_for_status()

cot = pd.read_csv(io.StringIO(r.text), sep='\t')
cot = cot.rename(columns={'From': 'query_id',
                          'Protein names': 'description',
                          'Gene Names': 'gene_name'})
cot['query_id'] = cot['query_id'].astype(str)
cot['crop'] = 'cotton'
cot = cot.drop_duplicates('query_id')

print(f'    {cot.query_id.nunique()}/{len(cot_ids)} genes matched')
cot.head(3)

[cotton] 239 genes -> UniProt idmapping
    239/239 genes matched


,query_id,description,gene_name,crop
0,107886209,"Chaperone protein ClpC1, chloroplastic",LOC107886209,cotton
1,107886412,DNA repair endonuclease UVH1 isoform X2,LOC107886412,cotton
3,107886866,Helicase-like transcription factor CHR28 isoform X3,LOC107886866,cotton


In [46]:
per_gene = pd.concat(up_tables + [cot[['crop', 'query_id', 'gene_name', 'description']]],
                     ignore_index=True)
per_gene = per_gene.drop_duplicates(subset=['crop', 'query_id'])

final = long.merge(per_gene, on=['crop', 'query_id'], how='left')
final = final[['peptide_id', 'peptide_seq', 'n_crops', 'crop',
               'gene_id', 'query_id', 'gene_name', 'description']]

print(f'{len(final)} peptide-gene pairs\n')
print('coverage by crop:')
print(final.groupby('crop').description
           .apply(lambda s: f'{s.notna().sum()}/{len(s)}').to_string())

1762 peptide-gene pairs

coverage by crop:
crop
chilli    334/334
cotton    286/286
grape       0/237
soy       687/687
tomato    218/218


In [41]:
from google.colab import files
up = files.upload()
GRAPE_FILE = list(up.keys())[0]

with open(GRAPE_FILE, errors='replace') as fh:
    head = [next(fh) for _ in range(5)]
print('--- raw first lines ---')
for l in head:
    print(repr(l[:180]))

sep = '\t' if '\t' in head[0] else ','
raw = pd.read_csv(GRAPE_FILE, sep=sep, nrows=5)
print('\n--- columns ---')
print(raw.columns.tolist())
raw

Saving PN40024.v4.1.REF.b2g.tsv to PN40024.v4.1.REF.b2g (1).tsv
--- raw first lines ---
'Sequence Name\tSequence Description\tSequence Length\tBlast Hits Count\tBlast Min E-Value\tBlast Similarity Mean\tBlast Top Hit Taxonomy Name\tBlast Top Hit Similarity\tAnnotation GO Count'
'Vitvi01g04000_t001\tRVW36801.1Retrovirus-related Pol polyprotein from transposon TNT 1-94\t189\t3\t1.4E-102\t76.87%\tVitis vinifera\t97.04%\t2\tGO:0003676;GO:0008270\t\tIPR001878;IPR001878;no'
'Vitvi01g04001_t001\tXP_034709496.1digalactosyldiacylglycerol synthase 2, chloroplastic\t230\t3\t1.8E-142\t91.74%\tVitis riparia\t91.74%\t3\tGO:0009707;GO:0019375;GO:0046481\tEC:2.4.1.241\tnoI'
'Vitvi01g04002_t001\tCBI27327.3unnamed protein product, partial\t126\t2\t1.8E-18\t89.60%\tVitis vinifera\t97.73%\t\t\t\t\t\n'
'Vitvi01g04003_t001\tRVW44210.1hypothetical protein CK203_089437\t103\t2\t4.0E-13\t83.02%\tVitis vinifera\t83.02%\t\t\t\t\t\n'

--- columns ---
['Sequence Name', 'Sequence Description', 'Sequence Length', 'B

,Sequence Name,Sequence Description,Sequence Length,Blast Hits Count,Blast Min E-Value,Blast Similarity Mean,Blast Top Hit Taxonomy Name,Blast Top Hit Similarity,Annotation GO Count,Annotation GO ID,Enzyme Code,InterPro Accession,InterPro GO ID
0,Vitvi01g04000_t001,RVW36801.1Retrovirus-related Pol polyprotein from transposon TNT 1-94,189,3,1.400000e-102,76.87%,Vitis vinifera,97.04%,2.0,GO:0003676;GO:0008270,NaN,IPR001878;IPR001878;noIPR;noIPR;noIPR;noIPR;IPR036875,GO:0008270/GO:0003676;GO:0008270/GO:0003676;;;;;GO:0008270/GO:0003676
1,Vitvi01g04001_t001,"XP_034709496.1digalactosyldiacylglycerol synthase 2, chloroplastic",230,3,1.800000e-142,91.74%,Vitis riparia,91.74%,3.0,GO:0009707;GO:0019375;GO:0046481,EC:2.4.1.241,noIPR;noIPR;noIPR;noIPR;IPR044525;noIPR;noIPR,;;;;GO:0046481;;
2,Vitvi01g04002_t001,"CBI27327.3unnamed protein product, partial",126,2,1.800000e-18,89.60%,Vitis vinifera,97.73%,NaN,NaN,NaN,NaN,NaN
3,Vitvi01g04003_t001,RVW44210.1hypothetical protein CK203_089437,103,2,4.000000e-13,83.02%,Vitis vinifera,83.02%,NaN,NaN,NaN,NaN,NaN
4,Vitvi01g04004_t001,XP_034695358.1uncharacterized protein LOC117921553,371,2,3.100000e-222,77.85%,Vitis riparia,71.19%,NaN,NaN,NaN,NaN,NaN


In [47]:
TSV = 'PN40024.v4.1.REF.b2g.tsv'          # uploaded at session root

b2g = pd.read_csv(TSV, sep='\t', usecols=['Sequence Name', 'Sequence Description'])

# transcript -> gene, uppercased to match your VITVI* IDs
b2g['query_id'] = (b2g['Sequence Name']
                   .str.replace(r'_t\d+$', '', regex=True)
                   .str.upper())

# strip the leading accession: "RVW36801.1Retrovirus-related..." -> "Retrovirus-related..."
b2g['description'] = (b2g['Sequence Description']
                      .astype(str)
                      .str.replace(r'^[A-Z]{2,3}[_]?\d{5,}\.\d+', '', regex=True)
                      .str.strip())

b2g['crop'] = 'grape'
b2g['gene_name'] = None
b2g = (b2g[['crop', 'query_id', 'gene_name', 'description']]
       .drop_duplicates('query_id'))

print(f'{len(b2g)} genes in the Blast2GO file')

# how many of yours are actually in it?
mine = set(long.loc[long.crop == 'grape', 'query_id'])
found = mine & set(b2g.query_id)
print(f'{len(found)}/{len(mine)} of your grape genes matched\n')
print(b2g[b2g.query_id.isin(list(found)[:3])].to_string(index=False))

35197 genes in the Blast2GO file
203/203 of your grape genes matched

 crop      query_id gene_name                                description
grape VITVI01G01723      None           unnamed protein product, partial
grape VITVI02G00919      None General transcription factor IIH subunit 2
grape VITVI06G00443      None        heat shock cognate 70 kDa protein 2


In [48]:
per_gene = pd.concat([per_gene, b2g], ignore_index=True).drop_duplicates(['crop', 'query_id'])

final = long.merge(per_gene, on=['crop', 'query_id'], how='left')
final = final[['peptide_id', 'peptide_seq', 'n_crops', 'crop',
               'gene_id', 'query_id', 'gene_name', 'description']]

print('coverage by crop:')
print(final.groupby('crop').description
           .apply(lambda s: f'{s.notna().sum()}/{len(s)}').to_string())

coverage by crop:
crop
chilli    334/334
cotton    286/286
grape     237/237
soy       687/687
tomato    218/218


In [49]:
gene_tbl = (final[['crop', 'gene_id', 'gene_name', 'description']]
            .drop_duplicates(subset=['crop', 'gene_id'])
            .sort_values(['crop', 'gene_id'])
            .reset_index(drop=True))

gene_tbl.to_csv('genes_with_descriptions.csv', index=False)
final.drop(columns=['query_id']).to_csv(
    'peptide_gene_descriptions.csv', index=False)

print(f'{len(gene_tbl)} unique genes')

from google.colab import files
files.download('genes_with_descriptions.csv')
files.download('peptide_gene_descriptions.csv')

gene_tbl

1467 unique genes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,crop,gene_id,gene_name,description
0,chilli,T459_00063,T459_00063,Luminal-binding protein 5
1,chilli,T459_00200,T459_00200,Cation/H(+) antiporter 17
2,chilli,T459_00327,T459_00327,Uncharacterized protein
3,chilli,T459_00328,T459_00328,Osmotin-like protein
4,chilli,T459_00399,T459_00399,Phospholipase D (EC 3.1.4.4)
...,...,...,...,...
1462,tomato,SOLYC12G040510.2,LOC101245737,Uncharacterized protein
1463,tomato,SOLYC12G057040.2,Cry1b,Cryptochrome 1b
1464,tomato,SOLYC12G088370.2,,Interferon-related developmental regulator N-terminal domain-containing protein
1465,tomato,SOLYC12G095900.2,LOC101268174,Protein EARLY FLOWERING 3
